# 04 - Feature engineering (new columns)

**Goal:** create new columns that may help the models, and **keep only the ones that raise the CV score**.

Method: train the same LightGBM (same settings, same folds) with and without each group of new columns, then compare.

| Group | New columns | Why |
|---|---|---|
| A. Floors | `income_is_floor`, `commute_is_floor` | Many rows sit at exactly 30000 income / 5.0 km. These may be a special group. |
| B. Totals & ratios | `total_stations`, `income_per_car`, `commute_x_stations` | Combine related columns. |
| C. Combos | `home_x_subsidy`, `city_x_car`, `anxiety_x_home` | Two text columns joined into one (e.g. "Yes_No"). |
| D. Numbers as categories | small number columns as text too | Lets trees treat each value on its own. |

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from config import *
from utils import *
import json
from train import train_lgbm, train_catboost

In [2]:
train, test, sample = load_data()
folds = get_folds(train)
y = train[TARGET].values

p = ROOT / "best_lgbm_params.json"
LGB_PARAMS = json.loads(p.read_text()) if p.exists() else {}
print("LightGBM params used:", LGB_PARAMS or "defaults")

LightGBM params used: {'num_leaves': 25, 'min_child_samples': 186, 'feature_fraction': 0.40542747440682847, 'bagging_fraction': 0.968336511497273, 'lambda_l1': 0.00368464480823872, 'lambda_l2': 0.3527941199754003, 'max_bin': 511, 'cat_smooth': 3.282184184531182}


In [3]:
def add_features(df):
    df = df.copy()
    new_num, new_cat = {}, {}
    # A. floors
    new_num["A"] = ["income_is_floor", "commute_is_floor"]
    df["income_is_floor"] = (df["Annual_Income_USD"] == 30000).astype(int)
    df["commute_is_floor"] = (df["Daily_Commute_km"] == 5.0).astype(int)
    # B. totals & ratios
    new_num["B"] = ["total_stations", "income_per_car", "commute_x_stations"]
    df["total_stations"] = df["Charging_Stations_Near_Home"] + df["Charging_Stations_Near_Work"]
    df["income_per_car"] = df["Annual_Income_USD"] / df["Number_of_Cars_Owned"]
    df["commute_x_stations"] = df["Daily_Commute_km"] / (1 + df["total_stations"])
    # C. combos of text columns
    new_cat["C"] = ["home_x_subsidy", "city_x_car", "anxiety_x_home"]
    df["home_x_subsidy"] = df["Home_Charging_Possible"] + "_" + df["Subsidy_Available"]
    df["city_x_car"] = df["City_Type"] + "_" + df["Current_Car_Type"]
    df["anxiety_x_home"] = df["Range_Anxiety_Level"] + "_" + df["Home_Charging_Possible"]
    # D. small number columns also as categories
    small = ["Number_of_Cars_Owned", "Charging_Stations_Near_Home",
             "Charging_Stations_Near_Work", "Environmental_Concern_Level"]
    new_cat["D"] = [f"{c}_cat" for c in small]
    for c in small:
        df[f"{c}_cat"] = df[c].astype(str)
    return df, new_num, new_cat

tr_fe, NEW_NUM, NEW_CAT = add_features(train)
te_fe, _, _ = add_features(test)

In [4]:
def run(groups, label):
    num = NUM_COLS + [c for g in groups for c in NEW_NUM.get(g, [])]
    cat = CAT_COLS + [c for g in groups for c in NEW_CAT.get(g, [])]
    X, X_test = to_category(tr_fe[num + cat], cat), to_category(te_fe[num + cat], cat)
    oof, test_pred = train_lgbm(X, y, X_test, folds, params=LGB_PARAMS, verbose=False)
    auc, ll = score(y, oof)
    print(f"{label:<12} AUC = {auc:.5f}   logloss = {ll:.5f}")
    return auc, oof, test_pred

results = {}
results["raw"] = run([], "raw")
for g in ["A", "B", "C", "D"]:
    results[g] = run([g], f"raw + {g}")

raw          AUC = 0.94280   logloss = 0.22473


raw + A      AUC = 0.94281   logloss = 0.22473


raw + B      AUC = 0.94260   logloss = 0.22509


raw + C      AUC = 0.94263   logloss = 0.22504


raw + D      AUC = 0.94258   logloss = 0.22513


## Keep the groups that helped
A group is kept if it beats the raw score. Then we train once more with all kept groups together.

In [5]:
base = results["raw"][0]
KEEP = [g for g in ["A", "B", "C", "D"] if results[g][0] > base]
print("Groups kept:", KEEP or "none")

if KEEP:
    auc, oof, test_pred = run(KEEP, "raw + " + "+".join(KEEP))
else:
    auc, oof, test_pred = results["raw"]
ll = score(y, oof)[1]
save_preds("lgbm_fe", oof, test_pred)
log_experiment("lgbm_fe", "raw+" + "+".join(KEEP) if KEEP else "raw", auc, ll, "lgbm with kept feature groups")
make_submission(test_pred, "lgbm_fe", sample);

Groups kept: ['A']


raw + A      AUC = 0.94281   logloss = 0.22473


Saved lgbm_fe.csv  rows=286571


## CatBoost with the kept columns
CatBoost is strong with combined text columns, so we also train it on the kept groups.

In [6]:
num = NUM_COLS + [c for g in KEEP for c in NEW_NUM.get(g, [])]
cat = CAT_COLS + [c for g in KEEP for c in NEW_CAT.get(g, [])]
oof, test_pred = train_catboost(tr_fe[num + cat], y, te_fe[num + cat], folds, cat_cols=cat, verbose=False)
auc, ll = score(y, oof)
print(f"CatBoost + FE  AUC = {auc:.5f}   logloss = {ll:.5f}")
save_preds("cat_fe", oof, test_pred)
log_experiment("cat_fe", "raw+" + "+".join(KEEP) if KEEP else "raw", auc, ll, "catboost with kept feature groups")
make_submission(test_pred, "cat_fe", sample);

CatBoost + FE  AUC = 0.94178   logloss = 0.22658


Saved cat_fe.csv  rows=286571


## TODO: add the original dataset
Playground data is made from a real dataset. Adding the real rows to training often helps.

- Search Kaggle Datasets for the original "EV purchase" data with these same columns.
- Download it into a new folder, e.g. `original_data/`.
- Add its rows to train **only inside each fold's training part**. Never put them in the validation part, or the CV score will be wrong.
- Keep it only if CV goes up.

*Not done yet: the source dataset has not been found.*

In [7]:
pd.read_csv(EXPERIMENTS_FILE).sort_values("cv_auc", ascending=False)

,model,features,cv_auc,cv_logloss,notes,date
4,lgbm_optuna,raw,0.942828,0.224683,"optuna 46 trials, lr 0.02",2026-10-17 15:59
5,lgbm_fe,raw+A,0.942806,0.224730,lgbm with kept feature groups,2026-10-17 16:12
2,xgb_v1,raw,0.941857,0.226431,default params,2026-10-17 15:11
6,cat_fe,raw+A,0.941778,0.226582,catboost with kept feature groups,2026-10-17 16:39
3,cat_v1,raw,0.941772,0.226594,default params,2026-10-17 15:39
1,lgbm_v1,raw,0.941771,0.226571,default params,2026-10-17 15:09
0,logreg_v1,raw,0.938094,0.233295,one-hot + scaling,2026-10-17 15:07
